SymPy implementation of the anaytical model for 3-dof training rig

#### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

import sympy as sp
import random
from datetime import datetime 

import json

In [ ]:
SAVE_EQNs_FLAG = False

#### System states

Choice of Euler-sequence: **3-2-1** sequence

\begin{equation*}
    \underline{\eta} = \begin{bmatrix} \eta_1 \\ \eta_2 \\ \eta_3 \end{bmatrix}
    \end{equation*}

based on **3-2-1** convention we follow standard notation and set:
- $\eta_1$ corresponds to yaw 
- $\eta_2$ corresponds to pitch
- $\eta_3$ corresponds to roll



In [ ]:
t = sp.symbols('t')

# Generalized Cordinates
n1 = sp.Function('n1')(t) # Yaw 
n2 = sp.Function('n2')(t) # Pitch
n3 = sp.Function('n3')(t) # Roll

# Generalized Cordinate Vector
n = sp.Matrix([n1, n2, n3]) # n
nd = n.diff(t) # ndot

## print check
# n
# nd

#### System parameters

1. **Acceleration due to gravity vector in $\mathcal{N}$ frame:**
\begin{equation*}
\underline{g}^{\mathcal{N}} = \begin{bmatrix} 0 \\ 0 \\ -g \end{bmatrix}^{\mathcal{N}}
\end{equation*}

2. **Geometric parameters:**
    
    Distances
    - $L_b$: Distance of the UAV-body from 3ODF arm pivot point
    - $L_c$: Distance of the compensator from 3ODF arm pivot point

    Moment of Inertia tensors (about $O_A$ expressed in $A$ frame components)
    - $I_a$: MOI Matrix of the Arm about 3DOF origin in arm-frame components
    - $I_b$: MOI Matrix of the UAV-body about 3DOF origin in arm-frame components (In principal axis: $I_{bx}$ $I_{by}$ $I_{bz}$)
    - $I_c$: MOI Matrix of the UAV-body about 3DOF origin in arm-frame components (In principal axis: $I_{cx}$ $I_{cy}$ $I_{cz}$)

3. **Material parameters:**
    - $m_a$: mass of the arm
    - $m_b$: mass of the UAV-body
    - $m_c$: mass of the compensator

In [ ]:
g, ma, mb, mc = sp.symbols('g ma mb mc')

# Legacy straight-arm length scalars kept for reporting/filename metadata
Lb, Lc = sp.symbols('Lb Lc')

In [ ]:
# Inertia tensors about O_A, expressed in arm frame
# Ia is full symmetric to support bent/non-axisymmetric arm geometry

#---Arm MOI---------------------------------
Iaxx, Iayy, Iazz    = sp.symbols('Iaxx Iayy Iazz')
Iaxy, Iaxz, Iayz    = sp.symbols('Iaxy Iaxz Iayz')

Ia = sp.Matrix([
    [Iaxx, Iaxy, Iaxz],
    [Iaxy, Iayy, Iayz],
    [Iaxz, Iayz, Iazz]
    ])

#---UAV-body MOI---------------------------------
Ibx, Iby, Ibz   = sp.symbols('Ibx Iby Ibz')
Ib              = sp.diag(Ibx, Iby, Ibz)

#---Compensator MOI---------------------------------
Icx, Icy, Icz   = sp.symbols('Icx Icy Icz')
Ic              = sp.diag(Icx, Icy, Icz)

#### Direction Cosine Matrix

- $C_{\mathcal{AN}}$: Matrix to express vectors in $\mathcal{A}$ frame from vectors in $\mathcal{N}$ frame

- $C_{\mathcal{AN}}^T = C_{\mathcal{NA}}$: Matrix to express vectors in $\mathcal{N}$ frame from vectors in $\mathcal{A}$ frame

In [ ]:
R1_n3 = sp.Matrix([
    [1, 0, 0],
    [0,  sp.cos(n3),   sp.sin(n3)],
    [0, -sp.sin(n3),   sp.cos(n3)]
])

R2_n2 = sp.Matrix([
    [sp.cos(n2), 0, -sp.sin(n2)],
    [0, 1, 0],
    [sp.sin(n2), 0, sp.cos(n2)]
])

R3_n1 = sp.Matrix([
    [sp.cos(n1),   sp.sin(n1), 0],
    [-sp.sin(n1),  sp.cos(n1), 0],
    [0, 0, 1]
])

# Cosie-Matrix: C_{AN}
# C_AN = R1_n3 * R2_n2 * R3_n1
# Hence C_AN.T -> is C_{NA}
C_AN = R1_n3 @ R2_n2 @ R3_n1 

# # print: C_AN
C_AN
# # print: C_NA
# C_AN.T

##### Sanity Check - 1

> - **Direction cosine-matrices are orthonormal**

\begin{align*}
    &C_{\mathcal{AN}} = C_{\mathcal{AN}}^T\\
    &C_{\mathcal{AN}} C_{\mathcal{AN}}^T = I
\end{align*}

> - **Unimodular property**

\begin{equation*}
    det(C_{\mathcal{AN}}) = 1
\end{equation*}

In [ ]:
# Check if direction cosine matrices are orthonormal 
sp.simplify(C_AN * C_AN.T)

In [ ]:
sp.simplify(C_AN.det())

#### Kineamatic Differential Equation

\begin{align*}
    \begin{bmatrix} \tilde{\omega} \end{bmatrix}^{Arm} &=
    \begin{bmatrix} C_{NA} \end{bmatrix}^{T} \begin{bmatrix} \dot{C}_{NA} \end{bmatrix}\\
    \begin{bmatrix} \tilde{\omega} \end{bmatrix}^{N(inertial)} &=
    \begin{bmatrix} C_{NA} \end{bmatrix} \begin{bmatrix} \tilde{\omega} \end{bmatrix}^{Arm} \begin{bmatrix} C_{NA} \end{bmatrix}^{T}\\ &= 
    \begin{bmatrix} \dot{C}_{NA} \end{bmatrix} \begin{bmatrix} C_{NA} \end{bmatrix}^{T}
\end{align*}

---

- **Angular velocity vector** 

\begin{equation*}
    \begin{bmatrix} \tilde{\omega} \end{bmatrix}^{any} = 
    \begin{bmatrix} 
        0 & -\omega_z & \omega_y \\
        \omega_z & 0 & -\omega_x \\
        -\omega_y & \omega_x & 0
    \end{bmatrix}
\end{equation*}

\begin{equation*}
    \mathbf{\omega}^{any} = 
    \begin{bmatrix}
        \begin{bmatrix} \tilde{\omega} \end{bmatrix}_{[2,1]} \\ \begin{bmatrix} \tilde{\omega} \end{bmatrix}_{[0,2]} \\ \begin{bmatrix} \tilde{\omega} \end{bmatrix}_{[1,0]}
    \end{bmatrix}
\end{equation*}

NOTE

> No matter the convecntion (3-2-1 or 1-2-3) as long as we change both $\underline{\eta}$ and the corresponding $C$ & $B$ matrix the interpretation of $\omega$ is always fixed
    
\begin{equation*}
    \underline{\omega} = \begin{bmatrix} \omega_x \\ \omega_y \\ \omega_z \end{bmatrix} = \begin{bmatrix} rotation-rate-x-axis \\ rotation-rate-y-axis \\ rotation-rate-z-axis \end{bmatrix}
\end{equation*}

In [ ]:
def get_omega_vec(S):
    return sp.Matrix([S[2,1], S[0,2], S[1,0]])

omega_skew_AN_A = -C_AN.diff(t) * C_AN.T
omega_AN_A      = get_omega_vec(omega_skew_AN_A)        # angular velocity of arm w.r.t to O_N written in arm-frame components
omega_AN_N      = C_AN.T * omega_AN_A                   # angular velocity of arm w.r.t to O_N written in inertial-frame components

# omega_A_AN
# omega_N_AN

In [ ]:
sp.simplify(omega_skew_AN_A - (C_AN * C_AN.diff(t).T))

##### Sanity Check - 2

> - **Skew-symmetric matrix property**

\begin{align*}
    &\begin{bmatrix} \~{\omega}_{\mathcal{A/N}} \end{bmatrix}_{\mathcal{A}} = - \begin{bmatrix} \~{\omega}_{\mathcal{A/N}} \end{bmatrix}_{\mathcal{A}}^T\\
    &\begin{bmatrix} \~{\omega}_{\mathcal{A/N}} \end{bmatrix}_{\mathcal{A}} + \begin{bmatrix} \~{\omega}_{\mathcal{A/N}} \end{bmatrix}_{\mathcal{A}}^T = I
\end{align*}

> - **get_omega_vec consistency**

Get back omega skew-symmetric matrix from omega-vector


> - **Constructing B-matrix**

\begin{align*}
    &\dot{\underline{\theta}} = \begin{bmatrix} B_{(\theta)} \end{bmatrix} \underline{\omega}_{\mathcal{AN}}\\
    &\underline{\omega}_{\mathcal{AN}} = \begin{bmatrix} B_{(\theta)} \end{bmatrix}^{-1} \dot{\underline{\theta}}_{\mathcal{AN}}
\end{align*}

Analyitical estimation of B-matrix

\begin{equation*}
    \begin{bmatrix} B_{(\theta)} \end{bmatrix}^{-1} = \frac{\partial \underline{\omega}_{AN}}{\partial \dot{\underline{\theta}}}
\end{equation*}

In [ ]:
# checking if the obtained matrix is skey-symmetric
skew_check1 = omega_skew_AN_A + omega_skew_AN_A.T
sp.simplify(skew_check1)

In [ ]:
def get_back_skew_symmetric(omega_vec):
    return sp.Matrix([
        [0, -omega_vec[2], omega_vec[1]],
        [omega_vec[2], 0, -omega_vec[0]],
        [-omega_vec[1], omega_vec[0], 0]
    ])

tildaomega_skew_AN_A = get_back_skew_symmetric(omega_AN_A)

sp.pprint(sp.simplify(tildaomega_skew_AN_A)==sp.simplify(omega_skew_AN_A))

In [ ]:
B_inv = omega_AN_A.jacobian(nd)               # omega_A_AN = B_inv * dn/dt
B_inv.simplify()
B_inv

#### Translational velocity

In [ ]:
# Arm-frame geometry points from 3DOF pivot O_A
# rO_A: arm COM position
# rB_A: body COM/attachment position
# rC_A: compensator COM/attachment position
xO_A, yO_A, zO_A    = sp.symbols('xO_A yO_A zO_A')
xB_A, yB_A, zB_A    = sp.symbols('xB_A yB_A zB_A')
xC_A, yC_A, zC_A    = sp.symbols('xC_A yC_A zC_A')

# Position vectors of COM/attachment points in arm-frame
rO_A    = sp.Matrix([xO_A, yO_A, zO_A])    # position vector of arm-CG from origin/3DOF joint in arm-frame components
rB_A    = sp.Matrix([xB_A, yB_A, zB_A])    # position vector of body-CG from origin/3DOF joint in arm-frame components
rC_A    = sp.Matrix([xC_A, yC_A, zC_A])    # position vector of compensator-CG from origin/3DOF joint in arm-frame componentsin arm-frame components

# Transform to inertial frame:
rO_N      = C_AN.T * rO_A
rB_N      = C_AN.T * rB_A
rC_N      = C_AN.T * rC_A

In [ ]:
# Translational velocity vectors of each body in inertial-frame components: omega x r
vO_cross = omega_AN_N.cross(rO_N)
vB_cross = omega_AN_N.cross(rB_N)
vC_cross = omega_AN_N.cross(rC_N)


# Total velocity
vO_N = rO_N.diff(t)
vB_N = rB_N.diff(t)
vC_N = rC_N.diff(t)

##### Sanity Check - 3

\begin{equation*}
    \frac{dr^{N}}{dt} = \frac{dr^{A}}{dt} \hspace{1pt} + \hspace{1pt} \omega \times r \hspace{1pt}
\end{equation*}

> - **Kineamatic Residual**: All translational velocity in arm-fram is zero

\begin{align*}
    \frac{dr^{N}}{dt} - \hspace{1pt} \omega \times r \hspace{1pt} = 0
\end{align*}

In [ ]:
# # Numerical sanity check for kinematics residual
# # (symbolic printouts may not look simplified to exact zeros)

# samples = []

# # Avoid dependence on global variable name `t`
# t_sym = next(iter(n1.free_symbols))

# kinematics_residual = sp.Matrix.vstack(
#     vO - vO_cross,
#     vB - vB_cross,
#     vC - vC_cross
# )
# for _ in range(5):
#     subs = {
#         n1: random.uniform(-1.2, 1.2),
#         n2: random.uniform(-1.0, 1.0),
#         n3: random.uniform(-1.2, 1.2),
#         sp.diff(n1, t_sym): random.uniform(-1.0, 1.0),
#         sp.diff(n2, t_sym): random.uniform(-1.0, 1.0),
#         sp.diff(n3, t_sym): random.uniform(-1.0, 1.0),
#         xO_A: -0.03,    yO_A: 0.00,     zO_A: -0.06,
#         xB_A: 0.00,     yB_A: 0.00,     zB_A: 0.12,
#         xC_A: 0.00,     yC_A: 0.00,     zC_A: -0.30,
#     }
#     r = kinematics_residual.subs(subs).evalf()
#     samples.append(max(abs(float(v)) for v in r))
    
# print('Max abs kinematics residual per sample:')
# print(samples)
# print('\nWorst-case residual:', max(samples))


In [ ]:
# Instead of checking using Kinematic Residual 
# we just simplify and check term by term equality:
print(vO_N.simplify() == vO_cross.simplify())
print(vB_N.simplify() == vB_cross.simplify())
print(vC_N.simplify() == vC_cross.simplify())

#### Kinematic Energy Terms

- **Translational Kinetic Energy**
\begin{align*}
    \frac{1}{2} \bigg( \sum_{i \in \{ a, b, c\}} m_i \begin{bmatrix} v_a \end{bmatrix}^T_{\mathcal{N}} \begin{bmatrix} v_a \end{bmatrix}_{\mathcal{N}} \bigg) 
\end{align*}

- **Rotational Kinetic Energy**
\begin{align*}
    \frac{1}{2} \bigg( \sum_{i \in \{ a, b, c\}} \begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix}^T_\mathcal{N}  \begin{bmatrix} I^a_{OA} \end{bmatrix}_{\mathcal{N}} \begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix}_\mathcal{N}  \bigg) 
\end{align*}


In [ ]:
T_trans = (
    sp.Rational(1,2)*ma*(vO_N.dot(vO_N)) +
    sp.Rational(1,2)*mb*(vB_N.dot(vB_N)) +
    sp.Rational(1,2)*mc*(vC_N.dot(vC_N))
)


T_rot = (
    sp.Rational(1,2)*(omega_AN_A.T * Ia * omega_AN_A)[0] + 
    sp.Rational(1,2)*(omega_AN_A.T * Ib * omega_AN_A)[0] +
    sp.Rational(1,2)*(omega_AN_A.T * Ic * omega_AN_A)[0]
)

# Every term is in inertial frame components
K = T_trans + T_rot

##### Sanity Check - 4

>**Energies are invariant under orthogonal rotations**

- Translational Kinetic Energy

\begin{align*}
    \begin{bmatrix} v_a \end{bmatrix}^T_{\mathcal{N}} \begin{bmatrix} v_a \end{bmatrix}_{\mathcal{N}} &= 
    \begin{bmatrix} v_a \end{bmatrix}^T_{\mathcal{A}} \Big( \begin{bmatrix} C_{\mathcal{NA}} \end{bmatrix}^T \begin{bmatrix} C_{\mathcal{NA}} \end{bmatrix} \Big) \begin{bmatrix} v_a \end{bmatrix}^T_{\mathcal{A}}\\
    &= \begin{bmatrix} v_a \end{bmatrix}^T_{\mathcal{A}} \begin{bmatrix} v_a \end{bmatrix}_{\mathcal{A}}
\end{align*}

- Rotational Kintetic Energy

\begin{align*}
    \begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix}^T_\mathcal{N}  \begin{bmatrix} I^a_{OA} \end{bmatrix}_{\mathcal{N}} \begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix}_\mathcal{N} 
    &=
    \Big(\begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix}^T_\mathcal{A} \begin{bmatrix} C_{\mathcal{NA}} \end{bmatrix}^T \Big)\hspace{4pt} 
    \Big(\begin{bmatrix} C_{\mathcal{NA}} \end{bmatrix} \begin{bmatrix} I^a_{OA} \end{bmatrix}_{\mathcal{A}} \begin{bmatrix} C_{\mathcal{NA}} \end{bmatrix}^T \Big) \hspace{4pt} 
    \Big(\begin{bmatrix} C_{\mathcal{NA}} \end{bmatrix} \begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix}_\mathcal{N} \Big)\\
    &=
    \begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix}^T_{\mathcal{A}}  \begin{bmatrix} I^a_{OA} \end{bmatrix}_{\mathcal{A}} \begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix}_\mathcal{A} 
\end{align*}


In [ ]:
# Velocity in A-frame
# Expected: (vN . vN) - (vA . vA) = 0
vO_A = C_AN * vO_N

expr = sp.simplify(vO_N.dot(vO_N) - vO_A.dot(vO_A))
sp.pprint(sp.simplify(expr))

In [ ]:
T_rot_test = (
    sp.Rational(1,2)*((omega_AN_N.T * C_AN.T) * (C_AN * Ia * C_AN.T) * (C_AN * omega_AN_N))[0] + 
    sp.Rational(1,2)*((omega_AN_N.T * C_AN.T) * (C_AN * Ib * C_AN.T) * (C_AN * omega_AN_N))[0] +
    sp.Rational(1,2)*((omega_AN_N.T * C_AN.T) * (C_AN * Ic * C_AN.T) * (C_AN * omega_AN_N))[0]
)

# If this is True then we verify that Energy is invariant under orthogonal rotations
print(sp.simplify(sp.Rational(1,2)*(omega_AN_A.T * Ia * omega_AN_A)[0]) ==  sp.simplify(sp.Rational(1,2)*((omega_AN_N.T * C_AN.T) * Ia * (C_AN * omega_AN_N))[0]))
print(sp.simplify(sp.Rational(1,2)*(omega_AN_A.T * Ib * omega_AN_A)[0]) ==  sp.simplify(sp.Rational(1,2)*((omega_AN_N.T * C_AN.T) * Ib * (C_AN * omega_AN_N))[0]))
print(sp.simplify(sp.Rational(1,2)*(omega_AN_A.T * Ic * omega_AN_A)[0]) ==  sp.simplify(sp.Rational(1,2)*((omega_AN_N.T * C_AN.T) * Ic * (C_AN * omega_AN_N))[0]))

>- **Zero velocity implied zero-KE**

Since $v$ & $\omega$ are functions of $\dot{\eta}$

if $\dot{\eta} = 0 \implies (v = \omega = 0)$ & $(K.E = 0)$

In [ ]:
# Zero-velocity check: if ndot = 0 => (v = omega = 0) & (K.E = 0)
subs_zero_vel_test = {
    sp.diff(n1,t): 0,
    sp.diff(n2,t): 0,
    sp.diff(n3,t): 0
}
sp.pprint(sp.simplify(K.subs(subs_zero_vel_test)))

>- **Validating in quadratic form of KE**

\begin{equation*}
    K_{total} = \frac{1}{2} \Big( \dot{\eta}^T \begin{bmatrix} M_{(\eta)} \end{bmatrix} \dot{\eta}\Big)
\end{equation*}

where $\begin{bmatrix} M_{(\eta)} \end{bmatrix}$ is called the conifuration-space inertia matrix (mass-matrix) & its properties:
- Symmertic:    
\begin{equation*} M_{(\eta)} = M_{(\eta)}^T \end{equation*}
- PSD:
\begin{equation} \dot{\eta}^T \begin{bmatrix} M_{(\eta)} \end{bmatrix} \dot{\eta} \geq 0 \end{equation}


In [ ]:
# Quadratic form in ndot: K = 1/2 * (ndot^T M(n) ndot) | M(n) must be 
# Symmetric & independent of ndot
nd_vec_test = sp.Matrix([sp.diff(n1,t), sp.diff(n2,t), sp.diff(n3,t)])

M_from_K_test = sp.hessian(K, nd_vec_test)

sp.pprint(sp.simplify(M_from_K_test - M_from_K_test.T))

In [ ]:
# Checking independence M(n) on ndot: 
# Indirectly reconstruct K.E back assuming the form: KE = 1/2 (ndot^T M(n) ndot)
K_reconstructed = sp.Rational(1,2) * (nd_vec_test.T * M_from_K_test * nd_vec_test)[0]

sp.pprint(sp.simplify(K - K_reconstructed))

In [ ]:
# M(n) should be PSD for rigid-body systems | But can be singular at Kinematic Sinularities
# Symbolically checking if M(n) is PSD -> Often return None & hard
# sp.Matrix(M_from_K_test).is_positive_definite

#### Potential Energy


\begin{equation*}
    P = - \sum_{i \in \{a, b, c\}} m_{i} \Big( \begin{bmatrix} g \end{bmatrix}^T_{\mathcal{N}} \begin{bmatrix} r_{i} \end{bmatrix}_{\mathcal{N}} \Big)
\end{equation*}

In [ ]:
# Acceleration due to gravity vector: g is in -ve Z-axis in inertial frame
gvec = sp.Matrix([0, 0, -g])

# Gravitational potential V satisfies F = -grad(V), with F = m*gvec
# hence V = -m*(gvec.dot(r))
P = -(
    ma*(gvec.dot(rO_N)) +
    mb*(gvec.dot(rB_N)) +
    mc*(gvec.dot(rC_N))
)

##### Sanity Check - 5

>- **Zero potential position**

If $\eta = \underline{0}$ then the potential energy should zero or then purely function of positions of COM

In [ ]:
# Zero-angle potential check
# Check: if (n1,n2,n3) = (0,0,0)rad then P.E should only be function of position of COM
# expected -> +ve (ma*za + mb*zb + mc*zc)
subs_zero = {n1:0, n2:0, n3:0}
sp.pprint(P.subs(subs_zero))

>- **Sign convention & dependence check**
Since 

$\begin{bmatrix} g \end{bmatrix}_{\mathcal{N}} = \begin{bmatrix} 0 & 0 & -g \end{bmatrix}^T_{\mathcal{N}}$

and

$\begin{bmatrix} r_a \end{bmatrix}_{\mathcal{N}} = \begin{bmatrix} x_a & y_a & z_a \end{bmatrix}^T_{\mathcal{N}}$

\begin{equation*}
    P_a = m_a \begin{bmatrix} g \end{bmatrix}^T_{\mathcal{N}} \begin{bmatrix} r^a \end{bmatrix}_{\mathcal{N}} = - m_a g z_a
\end{equation*}


Force due to gravity
\begin{equation*}
    \begin{bmatrix} \underline{F}^a_g \end{bmatrix}_{\mathcal{N}} = - \nabla_{\begin{bmatrix} r_a \end{bmatrix}_{\mathcal{N}}} P = \begin{bmatrix} 0 \\ 0 \\ +m_a g \end{bmatrix}
\end{equation*}


In [ ]:
# Force check for each mass
# Check: F = -grad(P)
# for body ma at (n1,n2,n3) = (0,0,0) -> expected (0, 0, -ma*g)
Fx = -sp.diff(P, xO_A)
Fy = -sp.diff(P, yO_A)
Fz = -sp.diff(P, zO_A)

# sp.simplify(Fx), sp.simplify(Fy), sp.simplify(Fz)
subs_zero = {n1:0, n2:0, n3:0}
F_eval = sp.Matrix([Fx, Fy, Fz]).subs(subs_zero)
print(F_eval)


# for body mb at (n1,n2,n3) = (0,0,0) -> expected (0, 0, -mb*g)
Fx = -sp.diff(P, xB_A)
Fy = -sp.diff(P, yB_A)
Fz = -sp.diff(P, zB_A)

# sp.simplify(Fx), sp.simplify(Fy), sp.simplify(Fz)
subs_zero = {n1:0, n2:0, n3:0}
F_eval = sp.Matrix([Fx, Fy, Fz]).subs(subs_zero)
print(F_eval)

# for body mc at (n1,n2,n3) = (0,0,0) -> expected (0, 0, -mc*g)
Fx = -sp.diff(P, xC_A)
Fy = -sp.diff(P, yC_A)
Fz = -sp.diff(P, zC_A)

# sp.simplify(Fx), sp.simplify(Fy), sp.simplify(Fz)
subs_zero = {n1:0, n2:0, n3:0}
F_eval = sp.Matrix([Fx, Fy, Fz]).subs(subs_zero)
print(F_eval)

>- **Invariance under yawing**

Performing any yaw shouldn't change the potential energy at all
\begin{equation}
    \frac{\partial P}{\partial \underline{\eta}} = \begin{bmatrix} 0 \\ non-zero \\ non-zero \end{bmatrix}
\end{equation}

In [ ]:
# Yaw invariance: gravity is invariant under yawing (n1 - by 3-2-1 convention).
# Expected --> 0
sp.pprint(sp.simplify(sp.diff(P, n1)))

In [ ]:
# Moments contributed by potential energy terms (Tg_i = dP/dn_i): Yaw invariance should still hold here and rest should be dependent
# Expected --> (0, nonzero, nonzero)
moment_dueto_gravity = sp.Matrix([
    sp.diff(P, n1),
    sp.diff(P, n2),
    sp.diff(P, n3)
])

sp.simplify(moment_dueto_gravity)
print("Gravity-torque due to yawing:\n", moment_dueto_gravity[0])
print("\nGravity-torque due to rolling:\n", moment_dueto_gravity[1])
print("\nGravity-torque due to pitching:\n", moment_dueto_gravity[2])

> - **Extracting Gravity-terms**

In [ ]:
# ----------- Extracting Gravity-term from Potential -----------
G_fromP = sp.Matrix([sp.diff(P, ni) for ni in n])

# Light cleanup only (DO NOT heavy simplify)
G_fromP = sp.expand(G_fromP)

print(r"$G_{(\eta)}$ extracted from Potential Energy.\n")
sp.simplify(G_fromP)

In [ ]:
# (d) Compare with your earlier gravity torque
print("\nCheck consistency with earlier moment_dueto_gravity:")
sp.pprint(sp.simplify(G_fromP - moment_dueto_gravity))

In [ ]:
# Independence from velocities
# Expected: Zero-matrix
print("Check ∂G/∂nd = 0:")
sp.pprint(G_fromP.jacobian(nd))

# Independence from accelerations
# Expected: Zero-matrix
ndd = sp.diff(nd, t)
print("\nCheck ∂G/∂ndd = 0:")
sp.pprint(G_fromP.jacobian(ndd))

In [ ]:
# Checking independence of Gravity-Forces (from Potential energy tems) w.r.t x & y, & only dependent on z or joint-angles
# Expected --> every-term should be dependent on n2 & n3 but not on n1--yaw in 3-2-1 convention (n1 purely changes x & y coordinates)
print("Potential energy dependence for arm:")
sp.pprint(sp.simplify(sp.diff(P, xO_A)))
sp.pprint(sp.simplify(sp.diff(P, yO_A)))
sp.pprint(sp.simplify(sp.diff(P, zO_A)))

print("\nPotential energy dependence for compensator:")
sp.pprint(sp.simplify(sp.diff(P, xC_A)))
sp.pprint(sp.simplify(sp.diff(P, yC_A)))
sp.pprint(sp.simplify(sp.diff(P, zC_A)))

print("\nPotential energy dependence for UAV:")
sp.pprint(sp.simplify(sp.diff(P, xB_A)))
sp.pprint(sp.simplify(sp.diff(P, yB_A)))
sp.pprint(sp.simplify(sp.diff(P, zB_A)))

>- **Point-function**: Rigid translation check

Since the potential energy terms are only function of $z_i$
\begin{equation*}
    P_{total}(z) = - \big( m_a g z_a + m_c g z_c + m_b g z_b \big)
\end{equation*}

Rigid-body translation by $dz$ should result in $dP$ ~ $dz$ 
\begin{equation*}
    P_{total} - P_{shift} = (m_a + m_c + m_b) g (dz)
\end{equation*}

<div class="alert alert-block">
    <b>Caution:</b> If shift is in body-frame components then there will be additional joint-angle terms 

In [ ]:
# Check: Rigid-body translation invariance
# Adding a constant offset to all bodies (dz) should change P by (dP ~ dz)
# Expected: dz * g * (ma + mb + mc) (additional n2,n3 terms for translation in A-frame)
dz = sp.symbols('dz')

P_shifted = P.subs({
    zO_A: zO_A + dz,
    zB_A: zB_A + dz,
    zC_A: zC_A + dz
})

sp.pprint(sp.simplify(P_shifted - P))

In [ ]:
# Total COM formulation equivalence: PE expressed about the total COM should be same as the sum of PE expressed for each body
# Expected: P_com - P_sum = 0
rG = (ma*rO_N + mb*rB_N + mc*rC_N)/(ma+mb+mc)
P_com = -(ma+mb+mc)*(gvec.dot(rG))

sp.pprint(sp.simplify(P - P_com))

#### Generalized Forces

\begin{align*}
    Q^{NC} &= Q_F + Q_{\tau}\\
    &= \sum_{i \in \{a, b, c\}} \Big( \begin{bmatrix} J_v \end{bmatrix}^T_i  \begin{bmatrix} \underline{F}_i \end{bmatrix} + \begin{bmatrix} J_{\omega} \end{bmatrix}^T_i  \begin{bmatrix} \underline{\tau}_i \end{bmatrix} \Big) \\
    &= \Big( \begin{bmatrix} J_v \end{bmatrix}^T_b  \begin{bmatrix} \underline{F}_b \end{bmatrix} \Big)^N + \Big( \begin{bmatrix} J_{\omega} \end{bmatrix}^T_b  \begin{bmatrix} \underline{\tau}_b \end{bmatrix} \Big)^A\\
    &= \Big( \begin{bmatrix} \frac{\partial [ v_b ] }{\partial \dot{\underline{\eta}}} \end{bmatrix}^T  \begin{bmatrix} \underline{F}_b \end{bmatrix} \Big)^N + \Big( \begin{bmatrix} \frac{\partial [ \omega_{\mathcal{AN}} ] }{\partial \dot{\underline{\eta}}} \end{bmatrix}^T  \begin{bmatrix} \underline{\tau}_b \end{bmatrix} \Big)^A
\end{align*}

We want both forces and torques are in $\mathcal{B}$-frame
\begin{align*}
    Q^{NC} &= \Big( \begin{bmatrix} J_v \end{bmatrix}^T_b  \begin{bmatrix} \underline{F}_b \end{bmatrix} \Big)^\mathcal{N} + \Big( \begin{bmatrix} J_{\omega} \end{bmatrix}^T_b  \begin{bmatrix} \underline{\tau}_b \end{bmatrix} \Big)^\mathcal{A}\\
    =& \Big( \begin{bmatrix} J_v \end{bmatrix}^T_b  \big( \begin{bmatrix} C_\mathcal{AN} \end{bmatrix}^T \begin{bmatrix} \underline{F}_b \end{bmatrix} \big)^\mathcal{A} \Big)^\mathcal{N} + \Big( \begin{bmatrix} J_{\omega} \end{bmatrix}^T_b  \begin{bmatrix} \underline{\tau}_b \end{bmatrix} \Big)^\mathcal{A}\\
    =& \begin{bmatrix} \Big( \begin{bmatrix} C_\mathcal{AN} \end{bmatrix}  \begin{bmatrix} J_{v} \end{bmatrix}_b \Big)^T & \begin{bmatrix} J_{\omega} \end{bmatrix}_b^T \end{bmatrix} 
    \begin{bmatrix} \begin{bmatrix} \underline{F}_b \end{bmatrix} \\ \\ \begin{bmatrix} \underline{\tau}_b \end{bmatrix}  \end{bmatrix}\\
    =& \begin{bmatrix} \tilde{J} \end{bmatrix}^{T} \begin{bmatrix} \mathcal{W} \end{bmatrix}
\end{align*}
 
- where $\begin{bmatrix} \tilde{J} \end{bmatrix} = \begin{bmatrix} \Big( \begin{bmatrix} C_\mathcal{AN} \end{bmatrix}  \begin{bmatrix} J_{v} \end{bmatrix}_b \Big)^T \\ \\ \begin{bmatrix} J_{\omega} \end{bmatrix}_b^T \end{bmatrix}$

- where net-wrench $\begin{bmatrix} \mathcal{W} \end{bmatrix} = \begin{bmatrix} \begin{bmatrix} \underline{F}_b \end{bmatrix} \\ \\ \begin{bmatrix} \underline{\tau}_b \end{bmatrix}  \end{bmatrix}$

*In $Q^{NC} NC$ stands for non-conservative forces.* 
- It represents generalized forces that cannot be derived from a potential energy function (like gravity or a spring).

-  Common examples of non-conservative forces:
    - Friction and air resistance (drag).
    - External driving forces (like a motor or someone pushing the system).
    - Time-varying inputs that add or remove energy from the system.

or 

\begin{align*}
    \Big(\begin{bmatrix} J_v \end{bmatrix}_b \Big)^{\mathcal{A}} &= \Big(- \begin{bmatrix} \tilde{r}^{b}_{\mathcal{OA}} \end{bmatrix} \begin{bmatrix} B_{(\eta)} \end{bmatrix}^{-1} \Big)^{\mathcal{A}}\\
    \Big(\begin{bmatrix} J_{\omega} \end{bmatrix}_b \Big)^{\mathcal{A}} &= \Big(\begin{bmatrix} B_{(\eta)} \end{bmatrix}^{-1} \Big)^{\mathcal{A}}

\end{align*}

> Note: 
\begin{align*}
    \Big( \begin{bmatrix} v_b \end{bmatrix} \Big)^{\mathcal{A}} &= \Big(\begin{bmatrix} J_v \end{bmatrix}_b \Big)^{\mathcal{A}} \underline{\dot{\eta}}\\
    \Big( \begin{bmatrix} \omega_{\mathcal{A/N}} \end{bmatrix} \Big)^{\mathcal{A}} &= \Big(\begin{bmatrix} J_{\omega} \end{bmatrix}_b \Big)^{\mathcal{A}} \underline{\dot{\eta}}
\end{align*}


**Alternative simplifed form**
\begin{align*}
    Q &= 
    \Big(
        - \begin{bmatrix} B_{(\eta)} \end{bmatrix}^{-T} \begin{bmatrix} \tilde{r}^{b}_{\mathcal{OA}} \end{bmatrix}^T \begin{bmatrix} \underline{F}_i \end{bmatrix} \hspace{4pt}
        + \hspace{4pt} \begin{bmatrix} B_{(\eta)} \end{bmatrix}^{-T}  \begin{bmatrix} \underline{\tau}_i \end{bmatrix}
    \Big)^{\mathcal{A}}\\
    &= 
    \begin{bmatrix} B_{(\eta)} \end{bmatrix}^{-T} 
    \Big(
        \begin{bmatrix} \underline{\tau}_i \end{bmatrix} \hspace{4pt}-\hspace{4pt} \begin{bmatrix} \tilde{r}^{b}_{\mathcal{OA}} \end{bmatrix}^T \begin{bmatrix} \underline{F}_i \end{bmatrix} 
    \Big)^{\mathcal{A}}\\
    &= 
    \begin{bmatrix} B_{(\eta)} \end{bmatrix}^{-T} 
    \Big(
        \begin{bmatrix} \underline{\tau}_i \end{bmatrix} \hspace{4pt}+\hspace{4pt} \begin{bmatrix} \tilde{r}^{b}_{\mathcal{OA}} \end{bmatrix} \begin{bmatrix} \underline{F}_i \end{bmatrix} 
    \Big)^{\mathcal{A}}\\
    &=  \begin{bmatrix} B_{(\eta)} \end{bmatrix}^{-T} \begin{bmatrix} \tau_{net-general} \end{bmatrix}^{\mathcal{A}}
\end{align*}




In [ ]:
# External wrench input (applied on point B (UAV-body)): (torque + force) in body-frame
tau1, tau2, tau3    = sp.symbols('tau1 tau2 tau3')
Fx, Fy, Fz          = sp.symbols('Fx Fy Fz')



# To enable forcing with numbers keep tau_A/F_A numeric for simulation.
# tau_b_A = sp.Matrix([0.1, 0, 0]) | F_b_A = sp.Matrix([0, 0, 0])
tau_b_A = sp.Matrix([tau1, tau2, tau3])
F_b_A   = sp.Matrix([Fx, Fy, Fz])

# Force and Torque jacobians:
# For torque it is easier in A-frame
Jw_b_B = omega_AN_A.jacobian(nd)  # or just B_inv
# For force it is easier either way: N-frame
Jv_b_N = vB_N.jacobian(nd) # or just rb_A_skew_test * B_inv

# Generalized force and torque contribution from force & torque applied at COM of B
Q_tau   = Jw_b_B.T * tau_b_A 

F_b_N   = C_AN.T * F_b_A
Q_force = Jv_b_N.T * F_b_N

# Every term is in inertial frame components
Q = Q_tau + Q_force

sp.simplify(Q)

In [ ]:
J_tildaTranspose_net = sp.Matrix.hstack(Jv_b_N.T * C_AN.T, Jw_b_B.T)

In [ ]:
J_tildaTranspose_net

In [ ]:
# J_tildaTranspose_net.nullspace()

##### Sanity Check - 7

>- **Jacobian from derivatie & from 3DOF simplification**

>- **Zero input check**

* Checklist

    - Zero force input check

    - Zero moment input check

    - Zero angle input check

>- **Virtual-work check**

>- **Independent of $\dot{\eta}$**

>- **Jacobian singularity check**

In [ ]:
rb_A_skew_test = get_back_skew_symmetric(rB_A)
rb_A_skew_test

In [ ]:
Jv_b_N_test = - rb_A_skew_test * B_inv

sp.simplify((C_AN * Jv_b_N) - Jv_b_N_test)

In [ ]:
sp.simplify(Jw_b_B - B_inv)

In [ ]:
# Zero-input check
# Expected: 0-vector
Q.subs({Fx:0, Fy:0, Fz:0, tau1:0, tau2:0, tau3:0})

In [ ]:
# Pure torque check
# Expected: B_inv^T * tau -> purely torque
Q_tau_only_test = Q.subs({Fx:0, Fy:0, Fz:0})
Q_tau_only_test

In [ ]:
# Pure force check
# Expected: B_inv^T * (r x F) -> pure force
Q_force_only_test = Q.subs({tau1:0, tau2:0, tau3:0})
Q_force_only_test

In [ ]:
# Symmetry / physical intuition check: if all joint angles are zero then the forces should be very simple
# Expected: Q = (r × F) + τ
Q.subs({n1:0, n2:0, n3:0})

In [ ]:
# Virtual work / power consistency
# Q^T ηdot = F^T v + τ^T ω
# Expected: 0 Joules
lhs = (Q.T * nd)[0]
rhs = (F_b_N.T * vB_N + tau_b_A.T * omega_AN_A)[0]

sp.simplify(lhs - rhs)

In [ ]:
# Independence from \eta_dot: Generalized forces must not depend on velocities
# Expected: Zero-matrix | If not --> inertial effects have been accidentally included
Q.jacobian(nd)

In [ ]:
# Singular configuration detection
# Expected: Zero at known Euler singularities | If near zero --> expect numerical issues (your earlier SVD issue)
sp.simplify(B_inv.det())

#### Euler-Lagrange equation - Matrix formulation

\begin{equation*}
    L = K - P
\end{equation*}

\begin{equation*}
    \frac{d}{dt} \bigg(\frac{d L}{d\underline{\dot{\eta}}} \bigg)  - \frac{dL}{d\underline{\eta}} = Q^{NC}
\end{equation*}

let
\begin{equation*}
    EL = \frac{d}{dt} \bigg(\frac{d L}{d\underline{\dot{\eta}}} \bigg)  - \frac{dL}{d\underline{\eta}} = 0
\end{equation*}


In [ ]:
# Lagrangian of the system
L = K - P


EL = []
for i, ni in enumerate(n):
    # dL/dn_dot
    dLdndot = sp.diff(L, sp.diff(ni, t))
    # d/dt(dL/dn_dot)
    d_dt = sp.diff(dLdndot, t)
    # dL/dn
    dLdn = sp.diff(L, ni)
    # d/dt(dL/dn_dot) - dL/dn = 0 -> EL Matrix formulation
    EL.append(d_dt - dLdn)

EL = sp.Matrix(EL)

print("\nEuler-Lagrange residual equations (EL = 0):")
sp.pprint(EL)

##### Sanity Check - 8

- **Extract mass-matrix & check it's properties**

    - $M_{(\eta)}$ Symmetric
    - $M_{(\eta)}$ extracted from Euler-Lagrage and directly from Kinetic energy should be exactly same
        - If same we already verified independence of $\dot{\eta}$

- **Extract remaining terms**: 

    $h(\eta, \dot{\eta}) = EL - M_{\eta} \ddot{\eta}$

    This terms should be independent of $\ddot{\eta}$

In [ ]:
# Checking for symmetric property
# Expected: Zero-matrix
M_fromEL = EL.jacobian(sp.diff(nd, t))   # mass matrix
sp.simplify(M_fromEL - M_fromEL.T)

In [ ]:
# Comparing with the mass-matrix extracted directly from K.E.
# Expected: Zero-matrix
sp.simplify(M_fromEL - M_from_K_test)

In [ ]:
# # Singular configuration behavior: At singularity:M becomes ill-conditioned

# sp.pprint(sp.simplify(B_inv.det()))
# sp.pprint(sp.simplify(M_fromEL.det()))

In [ ]:
# Recover standard form: EL=M(η)ηddot +h(η,ηdot) -> Ensure no hidden acceleration terms outside M
# Expected: Zero-matrix
h_fromEL = EL - M_fromEL * sp.diff(nd, t)

h_fromEL = sp.expand(h_fromEL)

# Derivative test can only find linear relation with \eta_ddot but nonlinear cannot be detected
h_fromEL.jacobian(sp.diff(nd, t))

In [ ]:
# # You can also just substitute \eta_ddot = 0 (assuming EL is linear in \etaddot)
# h_sym = EL.subs({sp.diff(n1,t,t): 0, sp.diff(n2,t,t): 0, sp.diff(n3,t,t): 0 })
# sp.simplify(h_fromEL - h_sym)
# # Expected -> zero vector

In [ ]:
print(h_fromEL.has(sp.diff(n1,t,t)))
print(h_fromEL.has(sp.diff(n2,t,t)))
print(h_fromEL.has(sp.diff(n3,t,t)))

> - **Extracting Corriolis & Centrifugal Term**

In [ ]:
#-------- C(q, qdot) EXTRACTION -------------
# Coriolis + centrifugal = h - G
C_fromEL = h_fromEL - G_fromP


# DO NOT fully simplify
# C_fromEL = sp.simplify(C_fromEL)
C_fromEL = sp.expand(C_fromEL)

In [ ]:
print(C_fromEL.has(sp.diff(n1,t,t)))
print(C_fromEL.has(sp.diff(n2,t,t)))
print(C_fromEL.has(sp.diff(n3,t,t)))

In [ ]:
print(G_fromP.has(sp.diff(n1,t,t)))
print(G_fromP.has(sp.diff(n2,t,t)))
print(G_fromP.has(sp.diff(n3,t,t)))

In [ ]:
# Check full reconstruction:
# h ?= C + G

residual = sp.simplify(h_fromEL - (C_fromEL + G_fromP))
    
print("Check h - (C + G) = 0:")
sp.pprint(residual)

In [ ]:
# (a) Should NOT depend on accelerations
print("Check ∂C/∂ndd = 0:")
sp.pprint(C_fromEL.jacobian(ndd))

In [ ]:
# (b) Should vanish when velocities are zero
zero_vel_subs = {
    sp.diff(n1,t): 0,
    sp.diff(n2,t): 0,
    sp.diff(n3,t): 0
}
print("\nCheck C(q,0) = 0:")
sp.pprint(sp.simplify(C_fromEL.subs(zero_vel_subs)))

#### Save derivation

In [ ]:
print("Simplifying Q-term")
Q = sp.simplify(Q)
print("Simplified Q-term\n")

print("Simplifying Mass-Matrix")
M = sp.simplify(M_fromEL)
print("Simplified Mass-Matrix\n")

# print("Simplifying h-term")
# h = sp.simplify(h_fromEL)
# print("Simplified h-term\n")

In [ ]:
def serialize(obj):
    # SymPy expressions
    if isinstance(obj, sp.Basic):
        return {"__sympy__": True, "expr": sp.srepr(obj)}
    
    # SymPy Matrix (explicit handling optional, but clearer)
    if isinstance(obj, sp.Matrix):
        return {"__sympy__": True, "expr": sp.srepr(obj)}
    
    # dict
    if isinstance(obj, dict):
        return {k: serialize(v) for k, v in obj.items()}
    
    # list / tuple
    if isinstance(obj, (list, tuple)):
        return [serialize(v) for v in obj]
    
    # fallback (numbers, strings)
    return obj



def deserialize(obj):
    if isinstance(obj, dict) and "__sympy__" in obj:
        return sp.sympify(obj["expr"])
    
    if isinstance(obj, dict):
        return {k: deserialize(v) for k, v in obj.items()}
    
    if isinstance(obj, list):
        return [deserialize(v) for v in obj]
    
    return obj


SYMPY_LOCALS = {name: getattr(sp, name) for name in dir(sp)}
SYMPY_LOCALS.update({
    "Derivative": sp.Derivative   # safety fallback
})

def deserialize_fast(obj):
    if isinstance(obj, dict) and "__sympy__" in obj:
        return eval(obj["expr"], SYMPY_LOCALS) #{"Symbol": sp.Symbol, "Matrix": sp.Matrix, **sp.__dict__})
    
    if isinstance(obj, dict):
        return {k: deserialize_fast(v) for k, v in obj.items()}
    
    if isinstance(obj, list):
        return [deserialize_fast(v) for v in obj]
    
    return obj

In [ ]:
# ============================================================
# 1. DEFINE SYMBOLS (numerical form)
# ============================================================
n1_sym, n2_sym, n3_sym = sp.symbols('n1 n2 n3')
n1d_sym, n2d_sym, n3d_sym = sp.symbols('n1d n2d n3d')
n1dd_sym, n2dd_sym, n3dd_sym = sp.symbols('n1dd n2dd n3dd')

# original function-of-time variables
n1f, n2f, n3f = n   # these are n1(t), n2(t), n3(t)

In [ ]:
# ============================================================
# 2. BUILD SUBSTITUTIONS
# ============================================================

# Replace derivatives
subs_derivatives = {
    sp.Derivative(n1f, t): n1d_sym,
    sp.Derivative(n2f, t): n2d_sym,
    sp.Derivative(n3f, t): n3d_sym,

    sp.Derivative(n1f, (t, 2)): n1dd_sym,
    sp.Derivative(n2f, (t, 2)): n2dd_sym,
    sp.Derivative(n3f, (t, 2)): n3dd_sym,
}

# Replace functions → plain symbols
subs_functions = {
    n1f: n1_sym,
    n2f: n2_sym,
    n3f: n3_sym,
}

In [ ]:
# ============================================================
# 3. CLEAN FUNCTION (robust)
# ============================================================

def force_replace_derivatives(expr):
    for d in expr.atoms(sp.Derivative):
        expr = expr.subs(d, d.doit())
    return expr

def clean(expr):
    expr = force_replace_derivatives(expr)
    expr = expr.subs(subs_derivatives)
    expr = expr.subs(subs_functions)
    expr = force_replace_derivatives(expr)
    return expr

In [ ]:
# -------- REMOVE DERIVATIVES --------
M_sub        = clean(M)
C_fromEL_sub = clean(C_fromEL)
G_fromP_sub  = clean(G_fromP)
h_fromEL_sub = clean(h_fromEL)
Q_sub        = clean(Q)

C_AN_sub        = clean(C_AN)
B_inv_sub       = clean(B_inv)
Jv_b_N_sub      = clean(Jv_b_N)
Jw_b_B_sub      = clean(Jw_b_B)
omega_AN_A_sub  = clean(omega_AN_A)

rO_A_sub = clean(rO_A)
rB_A_sub = clean(rB_A)
rC_A_sub = clean(rC_A)

rO_N_sub = clean(rO_N)
rB_N_sub = clean(rB_N)
rC_N_sub = clean(rC_N)

vO_N_sub = clean(vO_N)
vB_N_sub = clean(vB_N)
vC_N_sub = clean(vC_N)

K_sub = clean(K)
P_sub = clean(P)
L_sub = clean(L)

In [ ]:
# ============================================================
# 5. FIX n, nd, ndd (CRITICAL)
# ============================================================

n_clean   = sp.Matrix([n1_sym, n2_sym, n3_sym])
nd_clean  = sp.Matrix([n1d_sym, n2d_sym, n3d_sym])
ndd_clean = sp.Matrix([n1dd_sym, n2dd_sym, n3dd_sym])

In [ ]:
symbolic_data = {
    # Core dynamics
    'M': M_sub,
    'C': C_fromEL_sub,
    'G': G_fromP_sub,
    'h': h_fromEL_sub,
    'Q': Q_sub,

    # States
    'n': n_clean,
    'nd': nd_clean,
    'ndd': ndd_clean,   # Donot set it to: sp.diff(nd, t),

    # Kinematics
    'C_AN': C_AN_sub,
    'B_inv': B_inv_sub,
    'Jv_b_N':Jv_b_N_sub,
    'Jw_b_B':Jw_b_B_sub,
    'omega_A': omega_AN_A_sub,

    # Positions
    'rO_A': rO_A_sub,
    'rB_A': rB_A_sub,
    'rC_A': rC_A_sub,

    'rO_N': rO_N_sub,
    'rB_N': rB_N_sub,
    'rC_N': rC_N_sub,

    # Velocities (recommended addition)
    'vO_N': vO_N_sub,
    'vB_N': vB_N_sub,
    'vC_N': vC_N_sub,

    # Energies
    'K': K_sub,
    'P': P_sub,
    'L': L_sub,

    # Parameters
    'params': {
        'g': g, 'ma': ma, 'mb': mb, 'mc': mc,
        'Iaxx': Iaxx, 'Iayy': Iayy, 'Iazz': Iazz,
        'Iaxy': Iaxy, 'Iaxz': Iaxz, 'Iayz': Iayz,
        'Ibx': Ibx, 'Iby': Iby, 'Ibz': Ibz,
        'Icx': Icx, 'Icy': Icy, 'Icz': Icz,
        'xO_A': xO_A, 'yO_A': yO_A, 'zO_A': zO_A,
        'xB_A': xB_A, 'yB_A': yB_A, 'zB_A': zB_A,
        'xC_A': xC_A, 'yC_A': yC_A, 'zC_A': zC_A
    }
}

timestamp = datetime.now().strftime("%d%m%Y_%H%M")
filename = f"cache/3DOF_symbolic_data_{timestamp}.json"

In [ ]:
def check_no_derivatives(obj, name):
    if isinstance(obj, sp.Basic):
        assert not obj.has(sp.Derivative), f"{name} has Derivative"
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            check_no_derivatives(v, f"{name}[{i}]")
    elif isinstance(obj, dict):
        for k, v in obj.items():
            check_no_derivatives(v, f"{name}.{k}")



check_no_derivatives(symbolic_data, "symbolic_data")
print("No Derivatives anywhere ✓")

In [ ]:
print("FINAL CHECK:")
print("M contains accel:", any(v in M_sub.free_symbols for v in [n1dd_sym, n2dd_sym, n3dd_sym]))
print("h contains accel:", any(v in h_fromEL_sub.free_symbols for v in [n1dd_sym, n2dd_sym, n3dd_sym]))
print("Q contains accel:", any(v in Q_sub.free_symbols for v in [n1dd_sym, n2dd_sym, n3dd_sym]))

In [ ]:
if SAVE_EQNs_FLAG:
    with open(filename, "w") as f:
        json.dump(serialize(symbolic_data), f, indent=2)

    print(f"Saved to: {filename}")

> #### Sanity check: saved symbol-data

In [ ]:
with open(filename, "r") as f:
    loaded_raw = json.load(f)


assert loaded_raw == serialize(symbolic_data)
print("JSON exact match ✓")

In [ ]:
# data_loaded = deserialize(loaded_raw)
data_loaded = deserialize_fast(loaded_raw)

In [ ]:
print("Q check:", sp.simplify(data_loaded['Q'] - Q_sub) == sp.zeros(3,1))
print("C check:", sp.simplify(data_loaded['C'] - C_fromEL_sub) == sp.zeros(3,1))
print("G check:", sp.simplify(data_loaded['G'] - G_fromP_sub) == sp.zeros(3,1))
print("M check:", sp.simplify(data_loaded['M'] - M_sub) == sp.zeros(3,3))

In [ ]:
print("P check:", sp.simplify(data_loaded['P'] - P_sub) == 0)
print("K check:", sp.simplify(data_loaded['K'] - K_sub) == 0)
print("L check:", sp.simplify(data_loaded['L'] - L_sub) == 0)